In [6]:
# ── Colab Setup (run this first) ──
!pip install openai langgraph langchain-core pandas tabulate -q

import os

# Option 2: Use Colab Secrets (recommended)
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("AutoDataEngineer")

Notebook 02 — Agent Pipeline (THE CORE)
AutoDataEngineer: Self-Correcting Multi-Agent Data Cleaning

WHAT THIS DOES:
- Profiles messy data + generates a Data Contract
- Writes cleaning code autonomously
- Runs the code + validates against the contract
- If it fails → feeds error back → rewrites code (self-correcting loop)
- Outputs clean data + quality report

COLAB SETUP:
import os; os.environ["OPENAI_API_KEY"] = "sk-..."

PREREQUISITE: Run Notebook 01 first

### Setup

In [7]:
import os
import sys
import json
import time
import pandas as pd
from pathlib import Path

sys.path.insert(0, os.path.join(os.getcwd(), "..", "src"))
sys.path.insert(0, os.path.join(os.getcwd(), "src"))
sys.path.insert(0, "src")
sys.path.insert(0, "..")

DATA_DIR = Path("data")
if not DATA_DIR.exists():
    DATA_DIR = Path("../data")

# Load raw data
df_raw = pd.read_csv(DATA_DIR / "raw_data.csv")
print(f"Loaded raw data: {df_raw.shape}")

Loaded raw data: (200, 14)


### Initialize agents

In [9]:
from agents import ProfilerAgent, CoderAgent, QAAgent

profiler = ProfilerAgent()
coder = CoderAgent()
qa = QAAgent()

print("All agents initialized")

All agents initialized


### Run the Profiler Agent

In [10]:
print("=" * 60)
print("AGENT 1: PROFILER")
print("=" * 60)

profile_result = profiler.profile(df_raw)

# Display profiling report
report = profile_result.get("profiling_report", {})
print(f"\nSummary: {report.get('summary', 'N/A')}")
print(f"Issues found: {report.get('issues_found', 0)}")

print("\n-- Issues --")
for issue in report.get("issues", []):
    severity = issue.get("severity", "?").upper()
    print(f"  [{severity:6s}] {issue.get('column', '?'):25s} → {issue.get('issue', '')}")

# Display Data Contract
print("\n-- Data Contract --")
contract = profile_result.get("data_contract", {})
for col, rules in contract.items():
    if col.startswith("__"):
        continue
    action = rules.get("cleaning_action", "none")
    print(f"  {col:30s} → type={rules.get('expected_type','?'):8s} "
          f"nullable={str(rules.get('nullable','?')):5s} action={action}")

# Save profiling result
with open(DATA_DIR / "profile_result.json", "w") as f:
    json.dump(profile_result, f, indent=2, default=str)
print(f"\nProfile saved to {DATA_DIR / 'profile_result.json'}")

AGENT 1: PROFILER

Summary: The dataset exhibits several data quality issues, including missing values, inconsistent formats, and duplicates. These issues could impact data analysis and require attention to ensure data integrity.
Issues found: 10

-- Issues --
  [MEDIUM] product_name              → Null/missing values
  [MEDIUM] brands                    → Null/missing values and inconsistent casing
  [HIGH  ] categories                → Null/missing values and inconsistent casing
  [HIGH  ] quantity                  → Null/missing values and inconsistent formats
  [MEDIUM] serving_size              → Null/missing values and inconsistent formats
  [MEDIUM] energy_100g               → Null/missing values and mixed types
  [MEDIUM] nutrition_grade_fr        → Null/missing values
  [MEDIUM] countries                 → Null/missing values
  [HIGH  ] last_modified_datetime    → Inconsistent date formats
  [HIGH  ] __duplicates__            → Duplicate rows

-- Data Contract --
  product_nam

### Define LangGraph pipeline with self-correcting loop

In [11]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class PipelineState(TypedDict):
    """State flowing through the cleaning pipeline."""
    df_raw_csv: str                # Raw data as CSV string
    profile_result: dict           # From Profiler Agent
    current_code: str              # Latest code from Coder Agent
    attempt: int                   # Current attempt number
    error_history: list[dict]      # All previous errors (error memory)
    qa_result: dict                # Latest QA result
    df_cleaned_csv: str            # Cleaned data as CSV (final output)
    cleaning_log: dict             # What was cleaned
    processing_log: list[str]      # Human-readable log
    is_complete: bool              # Pipeline done flag


def generate_code_node(state: PipelineState) -> dict:
    """Coder Agent generates cleaning code."""
    log = list(state.get("processing_log", []))
    attempt = state.get("attempt", 0) + 1

    log.append(f"\n>>> CODER AGENT: Writing cleaning code (attempt {attempt})...")

    if attempt >= 3:
        log.append("    *** STRATEGY ESCALATION — trying fundamentally different approach ***")

    sample_csv = state["df_raw_csv"][:3000]  # First ~3000 chars as sample

    code = coder.generate_code(
        profiling_result=state["profile_result"],
        sample_csv=sample_csv,
        attempt=attempt,
        error_history=state.get("error_history", []),
    )

    log.append(f"    Generated {len(code.splitlines())} lines of code")
    return {
        "current_code": code,
        "attempt": attempt,
        "processing_log": log,
    }


def execute_and_validate_node(state: PipelineState) -> dict:
    """QA Agent executes code and validates against contract."""
    log = list(state.get("processing_log", []))
    log.append(f"\n>>> QA AGENT: Executing code (attempt {state['attempt']})...")

    # Load DataFrame from CSV string
    from io import StringIO
    df = pd.read_csv(StringIO(state["df_raw_csv"]))

    # Execute and validate
    contract = state["profile_result"].get("data_contract", {})
    qa_result = qa.execute_and_validate(
        code=state["current_code"],
        df=df,
        data_contract=contract,
    )

    error_history = list(state.get("error_history", []))

    if not qa_result["success"]:
        # Code crashed
        log.append(f"    EXECUTION FAILED: {qa_result['execution_error'][:200]}")
        error_history.append({
            "attempt": state["attempt"],
            "code": state["current_code"],
            "error": qa_result["execution_error"],
            "error_type": "execution",
        })
        return {
            "qa_result": {k: v for k, v in qa_result.items() if k != "df_cleaned"},
            "error_history": error_history,
            "processing_log": log,
            "is_complete": False,
        }

    elif not qa_result["validation_passed"] and qa_result["contract_violations"]:
        # Code ran but output violates contract
        violations_text = json.dumps(qa_result["contract_violations"], indent=2)
        log.append(f"    CODE RAN but {len(qa_result['contract_violations'])} contract violations:")
        for v in qa_result["contract_violations"]:
            log.append(f"      [{v.get('severity','?'):6s}] {v.get('column','?')}: {v.get('message','')}")

        error_history.append({
            "attempt": state["attempt"],
            "code": state["current_code"],
            "error": f"Contract violations:\n{violations_text}",
            "error_type": "validation",
        })

        # Check if violations are minor enough to accept
        high_severity = [v for v in qa_result["contract_violations"] if v.get("severity") == "high"]
        if len(high_severity) == 0:
            log.append("    No high-severity violations — accepting with warnings")
            df_cleaned_csv = qa_result["df_cleaned"].to_csv(index=False)
            return {
                "qa_result": {k: v for k, v in qa_result.items() if k != "df_cleaned"},
                "df_cleaned_csv": df_cleaned_csv,
                "cleaning_log": qa_result.get("cleaning_log", {}),
                "error_history": error_history,
                "processing_log": log,
                "is_complete": True,
            }

        return {
            "qa_result": {k: v for k, v in qa_result.items() if k != "df_cleaned"},
            "error_history": error_history,
            "processing_log": log,
            "is_complete": False,
        }

    else:
        # Success — code ran and passed contract
        log.append(f"    SUCCESS — code executed and passed contract validation")
        cleaning_log = qa_result.get("cleaning_log", {})
        log.append(f"    Rows: {cleaning_log.get('rows_before','?')} → {cleaning_log.get('rows_after','?')}")
        if cleaning_log.get("actions"):
            for action in cleaning_log["actions"][:10]:
                log.append(f"      • {action}")

        df_cleaned_csv = qa_result["df_cleaned"].to_csv(index=False)
        return {
            "qa_result": {k: v for k, v in qa_result.items() if k != "df_cleaned"},
            "df_cleaned_csv": df_cleaned_csv,
            "cleaning_log": cleaning_log,
            "processing_log": log,
            "is_complete": True,
        }


def should_retry(state: PipelineState) -> str:
    """
    Routing function: retry or finish?

    This implements the retry cap (max 3 attempts) that prevents
    the infinite loop described in the LinkedIn post.
    """
    if state.get("is_complete", False):
        return "finish"

    if state.get("attempt", 0) >= qa.MAX_RETRIES:
        # ═══ RETRY CAP HIT — stop and flag for human ═══
        log = list(state.get("processing_log", []))
        log.append(f"\n    RETRY CAP REACHED ({qa.MAX_RETRIES} attempts)")
        log.append("    Flagging for human review — autonomous cleaning incomplete")
        state["processing_log"] = log
        return "finish"

    return "retry"


def finish_node(state: PipelineState) -> dict:
    """Final assembly node."""
    log = list(state.get("processing_log", []))
    log.append("\n>>> ASSEMBLY: Pipeline complete")

    if state.get("df_cleaned_csv"):
        log.append("    Status: CLEANED DATA AVAILABLE")
    else:
        log.append("    Status: CLEANING FAILED — needs human intervention")

    return {"processing_log": log}


# ── Build the graph ──
workflow = StateGraph(PipelineState)

workflow.add_node("generate_code", generate_code_node)
workflow.add_node("execute_validate", execute_and_validate_node)
workflow.add_node("finish", finish_node)

workflow.add_edge(START, "generate_code")
workflow.add_edge("generate_code", "execute_validate")

# Conditional edge: retry or finish
workflow.add_conditional_edges(
    "execute_validate",
    should_retry,
    {"retry": "generate_code", "finish": "finish"},
)
workflow.add_edge("finish", END)

app = workflow.compile()

print("LangGraph pipeline compiled")
print("Flow: START → Generate Code → Execute & Validate ──┐")
print("                  ↑                                 │")
print("                  └─── retry (if failed) ───────────┘")
print("                                                    │")
print("                                              → Finish → END")

LangGraph pipeline compiled
Flow: START → Generate Code → Execute & Validate ──┐
                  ↑                                 │
                  └─── retry (if failed) ───────────┘
                                                    │
                                              → Finish → END


### Run the full pipeline

In [12]:
print("\n" + "=" * 70)
print("RUNNING FULL AutoDataEngineer PIPELINE")
print("=" * 70)

start_time = time.time()

result = app.invoke({
    "df_raw_csv": df_raw.to_csv(index=False),
    "profile_result": profile_result,
    "current_code": "",
    "attempt": 0,
    "error_history": [],
    "qa_result": {},
    "df_cleaned_csv": "",
    "cleaning_log": {},
    "processing_log": [],
    "is_complete": False,
})

elapsed = time.time() - start_time

# Print processing log
print("\n-- Processing Log --")
for line in result["processing_log"]:
    print(line)

print(f"\nTotal time: {elapsed:.1f}s")


RUNNING FULL AutoDataEngineer PIPELINE

-- Processing Log --

>>> CODER AGENT: Writing cleaning code (attempt 1)...
    Generated 119 lines of code

>>> QA AGENT: Executing code (attempt 1)...
    SUCCESS — code executed and passed contract validation
    Rows: 200 → 182
      • Filled nulls in product_name with 'Unknown'
      • Trimmed whitespace and standardized casing to lowercase in brands
      • Filled nulls in categories with 'Unknown' and standardized casing to lowercase
      • Filled nulls in quantity with 'Unknown' and standardized units to grams
      • Filled nulls in serving_size with 'Unknown' and standardized units to grams
      • Converted energy_100g to float and filled nulls with 0
      • Converted fat_100g to float and filled nulls with 0
      • Converted sugars_100g to float and filled nulls with 0
      • Converted proteins_100g to float and filled nulls with 0
      • Filled nulls in nutrition_grade_fr with 'unknown' and standardized casing to lowercase

>>>

<string>:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


<string>:19: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform

### Show the self-correcting loop log

In [13]:
print("\n" + "=" * 70)
print("SELF-CORRECTING LOOP LOG")
print("(The error memory + strategy escalation demo)")
print("=" * 70)

error_history = result.get("error_history", [])
if error_history:
    for entry in error_history:
        print(f"\n  Attempt {entry['attempt']} — {entry['error_type'].upper()} ERROR:")
        print(f"  {entry['error'][:300]}")
        if entry['attempt'] >= 3:
            print(f"  >> STRATEGY ESCALATION triggered on this attempt")
else:
    print("  No errors — code worked on first attempt!")

print(f"\n  Total attempts: {result.get('attempt', 0)}")
print(f"  Errors encountered: {len(error_history)}")
print(f"  Final status: {'SUCCESS' if result.get('is_complete') and result.get('df_cleaned_csv') else 'NEEDS HUMAN REVIEW'}")


SELF-CORRECTING LOOP LOG
(The error memory + strategy escalation demo)
  No errors — code worked on first attempt!

  Total attempts: 1
  Errors encountered: 0
  Final status: SUCCESS


### Show the generated cleaning code

In [14]:
print("\n" + "=" * 70)
print("GENERATED CLEANING CODE (AI-Written Python)")
print("=" * 70)
print(result.get("current_code", "No code generated"))


GENERATED CLEANING CODE (AI-Written Python)
cleaning_log = {"actions": [], "rows_before": len(df), "rows_after": 0}

# Cleaning product_name
try:
    df['product_name'].fillna('Unknown', inplace=True)
    cleaning_log["actions"].append("Filled nulls in product_name with 'Unknown'")
except Exception as e:
    cleaning_log["actions"].append(f"Error cleaning product_name: {e}")

# Cleaning brands
try:
    df['brands'] = df['brands'].str.strip().str.lower()
    cleaning_log["actions"].append("Trimmed whitespace and standardized casing to lowercase in brands")
except Exception as e:
    cleaning_log["actions"].append(f"Error cleaning brands: {e}")

# Cleaning categories
try:
    df['categories'].fillna('Unknown', inplace=True)
    df['categories'] = df['categories'].str.lower()
    cleaning_log["actions"].append("Filled nulls in categories with 'Unknown' and standardized casing to lowercase")
except Exception as e:
    cleaning_log["actions"].append(f"Error cleaning categories: {e}")

# Cl

### Display cleaned data vs raw data

In [15]:
if result.get("df_cleaned_csv"):
    from io import StringIO
    df_cleaned = pd.read_csv(StringIO(result["df_cleaned_csv"]))

    print("\n" + "=" * 70)
    print("BEFORE vs AFTER")
    print("=" * 70)

    print(f"\n  Raw:     {len(df_raw)} rows x {len(df_raw.columns)} cols")
    print(f"  Cleaned: {len(df_cleaned)} rows x {len(df_cleaned.columns)} cols")

    print(f"\n  Nulls before: {df_raw.isnull().sum().sum()}")
    print(f"  Nulls after:  {df_cleaned.isnull().sum().sum()}")

    print(f"\n  Duplicates before: {df_raw.duplicated().sum()}")
    print(f"  Duplicates after:  {df_cleaned.duplicated().sum()}")

    # Show cleaning log
    cl = result.get("cleaning_log", {})
    if cl.get("actions"):
        print(f"\n-- Cleaning actions performed --")
        for action in cl["actions"]:
            print(f"  • {action}")

    print(f"\n-- Sample cleaned rows --")
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 120)
    print(df_cleaned.head(10).to_string())

    # Save cleaned data
    clean_path = DATA_DIR / "cleaned_data.csv"
    df_cleaned.to_csv(clean_path, index=False)
    print(f"\nSaved cleaned data to {clean_path}")
else:
    print("\nNo cleaned data produced — check error log above")


BEFORE vs AFTER

  Raw:     200 rows x 14 cols
  Cleaned: 182 rows x 14 cols

  Nulls before: 416
  Nulls after:  193

  Duplicates before: 18
  Duplicates after:  0

-- Cleaning actions performed --
  • Filled nulls in product_name with 'Unknown'
  • Trimmed whitespace and standardized casing to lowercase in brands
  • Filled nulls in categories with 'Unknown' and standardized casing to lowercase
  • Filled nulls in quantity with 'Unknown' and standardized units to grams
  • Filled nulls in serving_size with 'Unknown' and standardized units to grams
  • Converted energy_100g to float and filled nulls with 0
  • Converted fat_100g to float and filled nulls with 0
  • Converted sugars_100g to float and filled nulls with 0
  • Converted proteins_100g to float and filled nulls with 0
  • Filled nulls in nutrition_grade_fr with 'unknown' and standardized casing to lowercase
  • Filled nulls in countries with 'Unknown' and standardized casing to uppercase
  • Parsed mixed date formats to I

### Pipeline statistics

In [16]:
print("\n" + "=" * 70)
print("PIPELINE STATISTICS")
print("=" * 70)

qa_res = result.get("qa_result", {})
violations = qa_res.get("contract_violations", [])

print(f"  Total attempts:           {result.get('attempt', 0)}")
print(f"  Errors encountered:       {len(error_history)}")
print(f"  Contract violations:      {len(violations)}")
print(f"  Validation passed:        {qa_res.get('validation_passed', False)}")
print(f"  Processing time:          {elapsed:.1f}s")

if result.get("df_cleaned_csv"):
    from io import StringIO
    df_c = pd.read_csv(StringIO(result["df_cleaned_csv"]))
    null_reduction = df_raw.isnull().sum().sum() - df_c.isnull().sum().sum()
    dup_reduction = df_raw.duplicated().sum() - df_c.duplicated().sum()
    print(f"  Nulls fixed:              {null_reduction}")
    print(f"  Duplicates removed:       {dup_reduction}")
    print(f"  Rows: {len(df_raw)} → {len(df_c)}")

print(f"\nPipeline complete!")


PIPELINE STATISTICS
  Total attempts:           1
  Errors encountered:       0
  Contract violations:      0
  Validation passed:        True
  Processing time:          34.9s
  Nulls fixed:              223
  Duplicates removed:       18
  Rows: 200 → 182

Pipeline complete!
